In [2]:
import pandas as pd
import numpy as np 
from sqlalchemy import create_engine
import os 
import glob

In [4]:
df = pd.read_csv('../Raw Data/olist_customers_dataset.csv')
df2 = pd.read_csv('../Raw Data/olist_products_dataset.csv')
print('file terbaca')
df2.info()

file terbaca
<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [ ]:
def batch_cleaner(data_configuration, Output ="Clean Data"):
    """
    Membersihkan banyak file sekaligus berdasarkan blueprint kolom masing-masing.
    """
    # Buat folder output jika belum ada
    os.makedirs(Output , exist_ok=True)
    
    # Looping membaca konfigurasi tiap file
    for rules_name, spek in data_configuration.items():
        # Cari semua file yang sesuai dengan path (bisa pakai wildcard seperti *.csv)
        file_list = glob.glob(spek['path'])
        
        if not file_list:
            print(f"no files found in path: {spek['path']}")
            continue
            
        for file_path in file_list:
            file_name = os.path.basename(file_path)
            print(f"... Processing: {file_name} ({rules_name})...")
            
            try:
                # 1. Load Data
                df = pd.read_csv(file_path)
                
                # 2. Pembersihan Wajib untuk semua file
                df.columns = df.columns.str.strip()
                df = df.drop_duplicates()
                
                # 3. Pembersihan Spesifik berbasis parameter
                if 'str_column' in spek and spek['str_column']:
                    # Pastikan kolom ada di file sebelum dibersihkan
                    str_target = [c for c in spek['str_column'] if c in df.columns]
                    if str_target:
                        df[str_target] = df[str_target].apply(lambda x: x.astype(str).str.lower().str.strip())
                
                if 'datetime_column' in spek and spek['datetime_column']:
                    for col in spek['datetime_column']:
                        if col in df.columns:
                            df[col] = pd.to_datetime(df[col], errors='coerce')
                
                if 'critical/id_column' in spek and spek['critical/id_column']:
                    target_id = [c for c in spek['critical/id_column'] if c in df.columns]
                    if target_id:
                        df = df.dropna(subset=target_id)
                
                # 4. Ekspor Hasil Bersih
                path_output = os.path.join(Output , f"cleaned_{file_name}")
                df.to_sql(path_output, index=False)
                print(f"saved on: {path_output}")
                
            except Exception as e:
                print(f"failed to process {file_name}. Error: {e}")
                
    print("\n Cleaning Done...")

In [ ]:
data_configuration = {
    'Aturan_Pelanggan': {
        'path': '../Raw Data/olist_customers_dataset.csv', # Bisa diisi satu file spesifik
        'str_column': ['customer_city'],
        'critical/id_column': ['customer_id', 'customer_unique_id']
    },
    'aturan_orders': {
        'path': '../Raw Data/olist_order*.csv',
        'datetime_column': [
            'shipping_limit_date','review_creation_date', 'review_answer_timestamp',
            "order_approved_at","order_delivered_carrier_date","order_delivered_customer_date",
            "order_estimated_delivery_date"],
        'critical/id_column': [
            'order_id', 'product_id', 'order_item_id', 'seller_id',
            'review_id', 'review_score', 'customer_id', 'order_delivered_customer_date'
            ]
    },
    'aturan_products': {
        'path': '../Raw Data/olist_products_dataset.csv',
        'str_column': '',
        'critical/id_column': ['product_id']
    }

}